REVERSE ENGINNERING CONSTITUENT CONTACTS

In [82]:
# Packages:
import geopandas as gdp
import pandas as pd

# Files:
address = gdp.read_file("/Users/sgaheer/Desktop/District 8 Data/Address/Address.shp")
districts = gdp.read_file("/Users/sgaheer/Desktop/District 8 Data/Council_District.shp")

# read column names:
# print(address.columns) # FullAddres

# read first 5 rows of FullAddres
#print(address['FullAddres'].head())

# subset districts
districts = districts[districts['DISTRICT'] == '8']
#print(districts.head())

#subset all address within D8
address = address[address.geometry.within(districts.geometry.iloc[0])]

# then, I just want the "FullAddres" column

#address = address[['FullAddres']]

#print(address.head())

# Then, export an excel sheet with a list of all the addresses
# with the column: FULL ADDRESS

address.to_excel('/Users/sgaheer/Desktop/District 8 Data/Address_District_8.xlsx')



In [83]:
# Now, put each address in their own neighborhood association, based on the neighborhoods shapefile
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

neighborhoods = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/Neighborhoods.kml", driver='KML')
print(neighborhoods.columns)

neighborhoods = neighborhoods.to_crs(address.crs)

# address dataframe from above

# spatial join
address_with_neighborhood = gdp.sjoin(address, neighborhoods, how="left", predicate="within")

address_with_neighborhood = address_with_neighborhood[['FullAddres', 'Name']]

print(address_with_neighborhood.head(10))

# Save to file
address_with_neighborhood.to_excel("/Users/sgaheer/Desktop/District 8 Data/addresses_with_neighborhoods.xlsx", index=False)




Index(['Name', 'Description', 'geometry'], dtype='object')
                FullAddres                                Name
20          1739 Quimby Rd                 Welch Park (Active)
21   1739 Ravens Place Way                           Dove Hill
22       1739 Rigoletto Dr            Brahms/Edgeview (Active)
103        174 Rio De Lata                           Dove Hill
141       1740 Barberry Ln                 Meadowfair (Active)
171      1740 Home Gate Dr                               Ocala
192         1740 Quimby Rd                 Welch Park (Active)
193  1740 Ravens Place Way                           Dove Hill
230    1740 Yerba Buena Rd  Linear Neighborhood (Silver Creek)
247    1741 Crane Ridge Ct  Linear Neighborhood (Silver Creek)


In [84]:
# Now, clean existing dataset.
# We want the names, phone number, email, and combine the addresses into a single column. 

# Load all three datasets.

a_l_data = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/Copy of Supports A-L.xlsx")
m_r_data = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/Copy of Supports M-R.xlsx")
s_z_data = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/Copy of Supports S-Z.xlsx")

#print(a_l_data.columns)
#print(m_r_data.columns)
#print(s_z_data.columns)

# combine rows so that the street address is in one column (row 3-6) is in one column seperated by a space
# if column 6 has "-", then combine columns 3-5

def combine_columns(row):
    if str(row[6]).strip() == "-":
        address_parts = [str(row[i]) for i in range(3, 6) if pd.notna(row[i])]
    else:
        address_parts = [str(row[i]) for i in range(3, 7) if pd.notna(row[i])]
    return " ".join(address_parts)

a_l_data['Address'] = a_l_data.apply(combine_columns, axis=1)
m_r_data['Address'] = m_r_data.apply(combine_columns, axis=1)
s_z_data['Address'] = s_z_data.apply(combine_columns, axis=1)

a_l_data = a_l_data[['Name', 'Phone_Number', 'Email', 'Address']]
m_r_data = m_r_data[['Name', 'Phone_Number', 'Email', 'Address']]
s_z_data = s_z_data[['Name', 'Phone_Number', 'Email', 'Address']]

# combine:

all_data = pd.concat([a_l_data, m_r_data, s_z_data], ignore_index=True)

# clean: 
all_data['Name'] = all_data['Name'].str.strip()
all_data['Phone_Number'] = all_data['Phone_Number'].str.strip()
all_data['Email'] = all_data['Email'].str.strip()
all_data['Address'] = all_data['Address'].str.strip()


# # Save to file
all_data.to_excel("/Users/sgaheer/Desktop/District 8 Data/all_data.xlsx", index=False)

# of course all_data will have duplicates! 



/var/folders/vq/97bmy6hd7pzfwr36kc4nlqbh0000gn/T/ipykernel_41612/1968316343.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if str(row[6]).strip() == "-":
/var/folders/vq/97bmy6hd7pzfwr36kc4nlqbh0000gn/T/ipykernel_41612/1968316343.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  address_parts = [str(row[i]) for i in range(3, 6) if pd.notna(row[i])]
/var/folders/vq/97bmy6hd7pzfwr36kc4nlqbh0000gn/T/ipykernel_41612/1968316343.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). 

In [85]:
# Now, cross reference the all_data dataset with the address_with_neighborhood dataset

# Load datasets
all_data = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/all_data.xlsx")
address_with_neighborhood = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/addresses_with_neighborhoods.xlsx")

# rename "FullAddres" to "Address"
address_with_neighborhood = address_with_neighborhood.rename(columns={'FullAddres': 'Address'})

# based on the address in all_data, find the neighborhood using address_with_neighborhood

# duplicates! more than one person lives in a house. 

merged_data = pd.merge(address_with_neighborhood, all_data, how='left', on='Address')

# Save to file
merged_data.to_excel("/Users/sgaheer/Desktop/District 8 Data/Neighborhoods.xlsx", index=False)

In [76]:
# count rows in all_data

#print(len(all_data))

# count rows in address_with_neighborhood

#print(len(address_with_neighborhood))

#print(len(merged_data))

# address_with_neighborhood = pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/addresses_with_neighborhoods.xlsx")

# duplicates = address_with_neighborhood[address_with_neighborhood.duplicated(subset=['FullAddres'], keep=False)]
# duplicates.to_excel("/Users/sgaheer/Desktop/District 8 Data/duplicates.xlsx", index=False) #33

# address = gdp.read_file("/Users/sgaheer/Desktop/District 8 Data/Address_District_8.xlsx")

# duplicates = address[address.duplicated(subset=['Field2'], keep=False)]
# print(duplicates)

# duplicates.to_excel("/Users/sgaheer/Desktop/District 8 Data/duplicates.xlsx", index=False)

duplicates = merged_data[merged_data.duplicated(subset=['Address'], keep=False)]

len(duplicates)

4813